# Import Brent daily price data
- Author: Bryan Bravo
- Created: 2026-03-18
## Import Libraries

In [1]:
import os
import sys

# Set JAVA_HOME before importing PySpark and use findspark
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-22'  # May need to remove or update in cloud environment.
import findspark
findspark.init()

import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)

# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

import proj_vars

### Initialize Spark Session


In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BusinessPlanAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.parquet.nativeio.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


## Variables

In [3]:
api_key = hardcoded_keys.EIA_API_KEY
end_date = proj_vars.end_date


## Custom Functions

In [4]:
def get_oil_data(col_name, series_name, end_date, api_key):
    try:
        print(f"Importing from EIA API {col_name.upper()}[{series_name}]")
        # Get first batch of data = dates {'2006-01-01' through '2024-12-31'}
        response = requests.get(
            f"https://api.eia.gov/v2/petroleum/pri/spt/data/?frequency=daily&data[0]=value&facets[series][]={series_name}"+
            "&start=2006-01-01&end=2024-12-31&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000"+
            f"&api_key={api_key}"
        )
        eia_data1 = response.json()

        # Get final batch of data {'2025-01-01' through provided value}
        response = requests.get(
            f"https://api.eia.gov/v2/petroleum/pri/spt/data/?frequency=daily&data[0]=value&facets[series][]={series_name}"+
            f"&start=2025-01-01&end={end_date}&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000"+
            f"&api_key={api_key}"
        )
        eia_data2 = response.json()

        # Create Pandas df and union datasets.
        df = (
            pd.concat([
                pd.DataFrame(eia_data1['response']['data'])[['period', 'value']],
                pd.DataFrame(eia_data2['response']['data'])[['period', 'value']]
                ], axis=0, ignore_index=True)
        )

        # Convert to PySpark and update variable names and dtypes
        df = (
            spark.createDataFrame(df)
            .withColumns({
                'date': F.date_format(F.to_date(F.col('period'), 'yyyy-MM-dd'), 'yyyyMMdd').cast('int'),
                f'{col_name}_dollars_per_barrel': F.col('value').cast('double')
            })
            .select('date', f'{col_name}_dollars_per_barrel')
        )
        print(f"Import for [{series_name}] Successful!\n************************************")
        return df
    except Exception as e:
        print(f"✗ Error fetching EIA data for {series_name}: {str(e)[:100]}")

## Query
### Import Crude Oil Price Data 

In [5]:
oil_dict = {
    'brent': 'RBRTE',
    'wti': 'RWTC'
}

globals().update({
    f"{oil_name}_df": get_oil_data(oil_name, series_name, end_date=end_date, api_key=api_key)
    for oil_name, series_name in oil_dict.items()
})

print("Joining and caching Spark DFs")
oil_df = brent_df.join(wti_df, on=['date'], how='inner')
oil_df.cache().count()

Importing from EIA API BRENT[RBRTE]
Import for [RBRTE] Successful!
************************************
Importing from EIA API WTI[RWTC]
Import for [RWTC] Successful!
************************************
Joining and caching Spark DFs


5033

In [6]:
oil_df.orderBy(F.desc('date')).show()

+--------+------------------------+----------------------+
|    date|brent_dollars_per_barrel|wti_dollars_per_barrel|
+--------+------------------------+----------------------+
|20260316|                  101.04|                 93.39|
|20260313|                  103.23|                 98.48|
|20260312|                  102.38|                 95.61|
|20260311|                   90.98|                  86.8|
|20260310|                   89.84|                 83.71|
|20260309|                   94.35|                 94.65|
|20260306|                   95.74|                 90.77|
|20260305|                   88.59|                 80.88|
|20260304|                   81.56|                 74.58|
|20260303|                   83.28|                 74.48|
|20260302|                   77.24|                 71.13|
|20260227|                   71.32|                 66.96|
|20260226|                   71.66|                  65.1|
|20260225|                   70.69|                  65.